In [1]:
!pip install transformers einops torch

  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached pygments-2.20.0-py3-none-any.whl.metadata (2.5 kB)
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.4 MB 329.7 kB/s eta 0:00:30
   --- ------------------------------------ 0.8/10.4 MB 488.4 kB/s eta 0:00:20
   ---- ----------------------------------- 1.0/10.4 MB 609.3 kB/s eta 0:00:16


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: D:\python\python.exe -m pip install --upgrade pip


In [2]:
from transformers import AutoTokenizer, AutoModel
import torch

# 指定模型 ID，使用 HyenaDNA 作为替代，因为 DNABERT-2 需要 triton 在 Windows 上难以安装
model_id = "LongSafari/hyenadna-large-1m-seqlen-hf"

# 加载分词器和模型（会自动下载到本地）
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModel.from_pretrained(model_id, trust_remote_code=True)

# 将模型设置为“评估模式”（不更新参数）
model.eval()

print("HyenaDNA 模型加载完毕！")

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--LongSafari--hyenadna-large-1m-seqlen-hf. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] A new version of the following files was downloaded from h

HyenaDNA 模型加载完毕！


In [3]:
# 1. 定义一段 DNA 序列（由 ATCG 组成）
dna_sequence = "ACGTAGCCTAG"

# 2. 将字符串转换为模型能理解的数字 (Token)
inputs = tokenizer(dna_sequence, return_tensors='pt')

# 3. 让模型进行计算（不计算梯度，节省内存）
with torch.no_grad():
    outputs = model(**inputs)

# 4. 获取所有词元的隐藏状态 (Hidden States)
# 根据论文，我们要对所有位置的输出取平均值，得到一个固定长度的向量
last_hidden_state = outputs[0]  # 形状为 [1, 序列长度, 隐藏维度]
embedding = torch.mean(last_hidden_state, dim=1)  # 形状变为 [1, 隐藏维度]

print("提取出的 Embedding 向量（前 10 位）：")
print(embedding[0][:10])
print("\n向量的总长度（维度）：", embedding.shape[1])

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


提取出的 Embedding 向量（前 10 位）：
tensor([-0.2783, -0.4107, -0.6189, -0.3789, -0.6084, -0.9816,  0.6831, -1.1731,
        -0.5084,  0.4838])

向量的总长度（维度）： 256
